In [ ]:
import os
import warnings
import numpy as np
import pandas as pd

from lifelines import CoxPHFitter
from lifelines.exceptions import ConvergenceError
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt

In [ ]:
def univariable_cox_with_zero_handling(
        df,
        feature_cols,
        duration_col="survival_time",
        event_col="event",
        censorship_col="censorship",
        drop_threshold=0.05,
        binary_threshold=0.20,
        min_group_n=10,
        min_group_events=3,
        penalizer=0.01,
        save_prefix="univariable_cox",
):

    data = df.copy()

    if event_col not in data.columns:
        if censorship_col not in data.columns:
            raise ValueError(f"Neither '{event_col}' nor '{censorship_col}' exists.")

        data[event_col] = (1 - pd.to_numeric(data[censorship_col], errors="coerce"))

    data[duration_col] = pd.to_numeric(data[duration_col], errors="coerce")
    data[event_col] = pd.to_numeric(data[event_col], errors="coerce")

    data.loc[data[duration_col] <= 0, duration_col] = np.nan

    valid_events = set(data[event_col].dropna().astype(int).unique())

    if not valid_events.issubset({0, 1}):
        raise ValueError(f"{event_col} must contain only 0 and 1, "f"but found {valid_events}.")

    result_rows = []
    excluded_rows = []

    for feature in feature_cols:
        if feature not in data.columns:
            excluded_rows.append({
                "feature": feature,
                "reason": "column_not_found",
            })
            continue

        tmp = data[[duration_col, event_col, feature]].copy()
        tmp[feature] = pd.to_numeric(tmp[feature], errors="coerce")
        tmp = tmp.replace([np.inf, -np.inf], np.nan).dropna()
        n_valid = len(tmp)
        if n_valid == 0:
            excluded_rows.append({
                "feature": feature,
                "reason": "no_valid_values",
            })
            continue

        x = tmp[feature]

        nonzero_mask = ~np.isclose(x.to_numpy(dtype=float), 0.0, atol=1e-12)
        nonzero_n = int(nonzero_mask.sum())
        nonzero_frac = nonzero_n / n_valid

        mean_value = float(x.mean())
        std_value = float(x.std())
        median_value = float(x.median())
        min_value = float(x.min())
        max_value = float(x.max())

        n_events = int(tmp[event_col].sum())

        if n_events == 0:
            excluded_rows.append({
                "feature": feature,
                "reason": "no_events",
                "n_valid": n_valid,
                "nonzero_fraction": nonzero_frac,
            })
            continue

        if x.nunique() < 2 or np.isclose(std_value, 0):
            excluded_rows.append({
                "feature": feature,
                "reason": "no_variation",
                "n_valid": n_valid,
                "nonzero_fraction": nonzero_frac,
            })
            continue


        if nonzero_frac < drop_threshold:
            excluded_rows.append({
                "feature": feature,
                "reason": "nonzero_fraction_below_drop_threshold",
                "n_valid": n_valid,
                "nonzero_n": nonzero_n,
                "nonzero_fraction": nonzero_frac,
                "mean": mean_value,
                "std": std_value,
            })
            continue

        if nonzero_frac < binary_threshold:
            analysis_type = "binary_presence"
            analysis_col = f"{feature}__present"
            tmp[analysis_col] = (~np.isclose(tmp[feature].to_numpy(dtype=float),0.0,atol=1e-12)).astype(int)

            group_summary = (tmp.groupby(analysis_col)[event_col].agg(["size", "sum"]))

            n_absent = int((tmp[analysis_col] == 0).sum())
            n_present = int((tmp[analysis_col] == 1).sum())

            events_absent = int(tmp.loc[tmp[analysis_col] == 0,event_col].sum())
            events_present = int(tmp.loc[tmp[analysis_col] == 1,event_col].sum())

            if min(n_absent, n_present) < min_group_n:
                excluded_rows.append({
                    "feature": feature,
                    "reason": "binary_group_too_small",
                    "n_valid": n_valid,
                    "nonzero_fraction": nonzero_frac,
                    "n_absent": n_absent,
                    "n_present": n_present,
                    "events_absent": events_absent,
                    "events_present": events_present,
                })
                continue

            if min(events_absent, events_present) < min_group_events:
                excluded_rows.append({
                    "feature": feature,
                    "reason": "too_few_events_in_binary_group",
                    "n_valid": n_valid,
                    "nonzero_fraction": nonzero_frac,
                    "n_absent": n_absent,
                    "n_present": n_present,
                    "events_absent": events_absent,
                    "events_present": events_present,
                })
                continue

            cox_df = tmp[[duration_col, event_col, analysis_col]].copy()

            interpretation = ("Present (>0) versus absent (=0)")


        else:
            analysis_type = "continuous_zscore"
            analysis_col = f"{feature}__z"
            tmp[analysis_col] = (tmp[feature] - tmp[feature].mean()) / tmp[feature].std(ddof=1)

            cox_df = tmp[[duration_col, event_col, analysis_col]].copy()

            n_absent = int((~nonzero_mask).sum())
            n_present = nonzero_n
            events_absent = np.nan
            events_present = np.nan

            interpretation = ("Hazard ratio per 1-SD increase")

        cph = CoxPHFitter(penalizer=penalizer)

        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")

                cph.fit(
                    cox_df,
                    duration_col=duration_col,
                    event_col=event_col,
                    show_progress=False,
                )

            row = cph.summary.loc[analysis_col]

            coef = float(row["coef"])
            hr = float(row["exp(coef)"])
            ci_low = float(row["exp(coef) lower 95%"])
            ci_high = float(row["exp(coef) upper 95%"])
            p_value = float(row["p"])
            se_coef = float(row["se(coef)"])
            z_value = float(row["z"])

            result_rows.append({
                "feature": feature,
                "analysis_variable": analysis_col,
                "analysis_type": analysis_type,
                "interpretation": interpretation,

                "n": n_valid,
                "events": n_events,

                "nonzero_n": nonzero_n,
                "nonzero_fraction": nonzero_frac,

                "zero_n": n_valid - nonzero_n,
                "zero_fraction": 1 - nonzero_frac,

                "n_absent": n_absent,
                "n_present": n_present,
                "events_absent": events_absent,
                "events_present": events_present,

                "mean_original": mean_value,
                "std_original": std_value,
                "median_original": median_value,
                "min_original": min_value,
                "max_original": max_value,

                "coef": coef,
                "se_coef": se_coef,
                "z": z_value,

                "hazard_ratio": hr,
                "ci95_low": ci_low,
                "ci95_high": ci_high,
                "p_value": p_value,
            })

        except (ConvergenceError, np.linalg.LinAlgError, ValueError, ZeroDivisionError) as exc:
            excluded_rows.append({
                "feature": feature,
                "reason": "cox_fit_failed",
                "error": str(exc),
                "n_valid": n_valid,
                "nonzero_fraction": nonzero_frac,
                "analysis_type": analysis_type,
            })


    results_df = pd.DataFrame(result_rows)
    excluded_df = pd.DataFrame(excluded_rows)

    if not results_df.empty:
        reject, p_adj, _, _ = multipletests(results_df["p_value"].to_numpy(), alpha=0.05, method="fdr_bh")
        results_df["p_adj_BH"] = p_adj
        results_df["significant_FDR"] = reject

        results_df["hr_ci95"] = results_df.apply(
            lambda row: (
                f"{row['hazard_ratio']:.3f} "
                f"({row['ci95_low']:.3f}–"
                f"{row['ci95_high']:.3f})"
            ),
            axis=1
        )

        results_df["p_report"] = results_df["p_value"].apply(format_p_value)
        results_df["fdr_report"] = results_df["p_adj_BH"].apply(format_p_value)
        results_df = results_df.sort_values(["p_adj_BH", "p_value"]).reset_index(drop=True)


    results_path = f"{save_prefix}_results.csv"
    excluded_path = f"{save_prefix}_excluded.csv"

    results_df.to_csv(results_path, index=False)
    excluded_df.to_csv(excluded_path, index=False)

    print("\n=== Univariable Cox results ===")
    if results_df.empty:
        print("No valid Cox results.")
    else:
        display_cols = [
            "feature",
            "analysis_type",
            "n",
            "events",
            "nonzero_fraction",
            "hazard_ratio",
            "ci95_low",
            "ci95_high",
            "p_value",
            "p_adj_BH",
            "significant_FDR",
        ]

        print(results_df[display_cols].round(4).to_string(index=False))

    print("\n=== Excluded features ===")
    if excluded_df.empty:
        print("None")
    else:
        print(excluded_df[["feature", "reason"]].to_string(index=False))

    print(f"\nSaved results:  {results_path}")
    print(f"Saved excluded: {excluded_path}")

    return results_df, excluded_df


def format_p_value(p):
    if pd.isna(p):
        return "NA"
    if p < 0.001:
        return "<0.001"
    return f"{p:.3f}"

In [ ]:
df = pd.read_csv("/root/Desktop/data/private/hjx_product/results_pgexplainer_0614/CombinedModel_ss/statistic_analysis/feature_selection_for_cox.csv")

if "event" not in df.columns:
    df["event"] = 1 - df["censorship"]

feature_cols = [
    "Necrosis_ratio",
    "Normal_ratio",
    "Reactive_ratio",

    "subgraph_Inflammation_pixel_ratio",
    "subgraph_Necrosis_pixel_ratio",
    "subgraph_Normal_pixel_ratio",
    "subgraph_Reactive_pixel_ratio",

    "Fibrous_delta",
    "Inflammation_delta",
    "Normal_delta",

    "Tumor__Fibrous_edge_weight_ratio",
    "Tumor__Necrosis_edge_ratio",
    "Tumor__Necrosis_edge_weight_ratio",
    "Tumor__Steatosis_edge_weight_ratio",
    "Fibrous__Normal_edge_ratio",
    "Fibrous__Reactive_edge_ratio",
    "Fibrous__Reactive_edge_weight_ratio",
    "Necrosis__Necrosis_edge_ratio",
    "Necrosis__Necrosis_edge_weight_ratio",
    "Necrosis__Steatosis_edge_ratio",
    "Necrosis__Steatosis_edge_weight_ratio",
    "Normal__Normal_edge_ratio",
    "Steatosis__Steatosis_edge_ratio",
    "Steatosis__Steatosis_edge_weight_ratio",
]

In [ ]:
save_dir = "/root/Desktop/data/private/hjx_product/results_pgexplainer_0614/CombinedModel_ss/statistic_analysis/cox_analysis/"
cox_results, excluded_features = univariable_cox_with_zero_handling(
    df=df,
    feature_cols=feature_cols,
    duration_col="survival_time",
    event_col="event",
    censorship_col="censorship",
    drop_threshold=0.05,
    binary_threshold=0.20,
    min_group_n=10,
    min_group_events=3,
    penalizer=0.01,
    save_prefix=save_dir,
)

In [ ]:
import os
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from lifelines import CoxPHFitter
from lifelines.exceptions import ConvergenceError


input_path = "/root/Desktop/data/private/hjx_product/results_pgexplainer_0614/CombinedModel_ss/statistic_analysis/feature_selection_for_cox.csv"
save_dir = "/root/Desktop/data/private/hjx_product/results_pgexplainer_0614/CombinedModel_ss/statistic_analysis/cox_analysis/"
os.makedirs(save_dir, exist_ok=True)

duration_col = "survival_time"
event_col = "event"
censorship_col = "censorship"

age_col = "age"
sex_col = "sex"
stage_col = "stage"

risk_col = "risk"
edge_col = "Tumor__Necrosis_edge_ratio"


def read_table(path):
    ext = os.path.splitext(path)[1].lower()

    if ext in [".xlsx", ".xls"]:
        return pd.read_excel(path)

    if ext == ".csv":
        return pd.read_csv(path)

    raise ValueError(f"Unsupported file type: {ext}")


def format_p(p):
    if pd.isna(p):
        return "NA"
    if p < 0.001:
        return "<0.001"
    return f"{p:.3f}"


def zscore(series):
    x = pd.to_numeric(series, errors="coerce")

    mean = x.mean()
    std = x.std(ddof=1)

    if pd.isna(std) or np.isclose(std, 0):
        raise ValueError(f"Cannot standardize '{series.name}': SD is zero or invalid.")

    return (x - mean) / std, mean, std


def clean_stage(value):
    if pd.isna(value):
        return np.nan

    text = str(value).strip().upper()
    missing_values = {
        "",
        "0",
        "NAN",
        "NONE",
        "UNKNOWN",
        "NOT REPORTED",
        "NOT AVAILABLE",
        "NA",
        "N/A",
    }

    if text in missing_values:
        return np.nan

    text = text.replace("PATHOLOGIC", "")
    text = text.replace("CLINICAL", "")
    text = text.replace("STAGE", "")
    text = text.strip()

    match = re.match(r"^(IV|III|II|I)", text)

    if match is None:
        return np.nan

    return match.group(1)


def fit_cox(
    df,
    covariates,
    model_name,
    display_names,
    penalizer=0.0,
):

    model_df = df[[duration_col, event_col] + covariates].copy()
    model_df = model_df.replace([np.inf, -np.inf], np.nan).dropna()
    model_df = model_df[model_df[duration_col] > 0].copy()

    n = len(model_df)
    events = int(model_df[event_col].sum())

    if n == 0:
        raise ValueError(f"No valid samples for model: {model_name}")

    if events == 0:
        raise ValueError(f"No observed events for model: {model_name}")

    print("\n" + "=" * 70)
    print(model_name)
    print(f"N = {n}")
    print(f"Events = {events}")
    print(f"Covariates = {covariates}")

    cph = CoxPHFitter(penalizer=penalizer)

    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")

            cph.fit(
                model_df,
                duration_col=duration_col,
                event_col=event_col,
                show_progress=False
            )

    except ConvergenceError as exc:
        raise RuntimeError(f"Cox model did not converge: {model_name}\n{exc}") from exc

    rows = []

    for var in covariates:
        row = cph.summary.loc[var]

        rows.append({
            "model": model_name,
            "variable": var,
            "display_name": display_names.get(var, var),

            "n": n,
            "events": events,

            "coef": float(row["coef"]),
            "se_coef": float(row["se(coef)"]),

            "hazard_ratio": float(row["exp(coef)"]),
            "ci95_low": float(row["exp(coef) lower 95%"]),
            "ci95_high": float(row["exp(coef) upper 95%"]),

            "z": float(row["z"]),
            "p_value": float(row["p"]),

            "concordance_index": float(cph.concordance_index_),
            "partial_AIC": float(cph.AIC_partial_),
            "log_likelihood": float(cph.log_likelihood_),
        })

    result_df = pd.DataFrame(rows)

    result_df["HR_95CI"] = result_df.apply(
        lambda r: (
            f"{r['hazard_ratio']:.3f} "
            f"({r['ci95_low']:.3f}–{r['ci95_high']:.3f})"
        ),
        axis=1
    )

    result_df["p_report"] = result_df["p_value"].apply(format_p)

    print(
        result_df[
            [
                "display_name",
                "hazard_ratio",
                "ci95_low",
                "ci95_high",
                "p_value",
            ]
        ].round(4).to_string(index=False)
    )

    return cph, result_df, model_df


def check_ph(cph, model_df, model_name):
    print("\nChecking PH assumption:", model_name)

    try:
        cph.check_assumptions(model_df, p_value_threshold=0.05, show_plots=False)
    except Exception as exc:
        print(f"[Warning] PH assumption check failed "f"for {model_name}: {exc}")

df = read_table(input_path)

print("Original N:", len(df))
print(df.columns.tolist())

df[duration_col] = pd.to_numeric(df[duration_col], errors="coerce")

if event_col not in df.columns:
    if censorship_col not in df.columns:
        raise ValueError(f"Neither '{event_col}' nor "f"'{censorship_col}' exists.")

    df[event_col] = (1- pd.to_numeric(df[censorship_col],errors="coerce"))

df[event_col] = pd.to_numeric(df[event_col],errors="coerce")
df.loc[df[duration_col] <= 0, duration_col] = np.nan

event_values = set(df[event_col].dropna().astype(int).unique())

if not event_values.issubset({0, 1}):
    raise ValueError(f"Event must be 0/1. Found: {event_values}")


df[age_col] = pd.to_numeric(df[age_col], errors="coerce")
df["age_per10"] = (df[age_col] / 10.0)
df[sex_col] = (df[sex_col].astype("string").str.strip().str.title())
df["sex_male"] = df[sex_col].map({"Female": 0,"Male": 1})
df["stage_main"] = (df[stage_col].apply(clean_stage))
df["stage_group"] = (df["stage_main"].map({
        "I": "Early",
        "II": "Early",
        "III": "Advanced",
        "IV": "Advanced",
    })
)

df["stage_advanced"] = (df["stage_group"].map({
        "Early": 0,
        "Advanced": 1,
    })
)


df["risk_z"], risk_mean, risk_std = zscore(df[risk_col])
df["tumor_necrosis_edge_z"], edge_mean, edge_std = zscore(df[edge_col])

scaling_df = pd.DataFrame([
    {
        "variable": risk_col,
        "mean": risk_mean,
        "std": risk_std,
        "transformed_variable": "risk_z",
    },
    {
        "variable": edge_col,
        "mean": edge_mean,
        "std": edge_std,
        "transformed_variable": (
            "tumor_necrosis_edge_z"
        ),
    },
])

scaling_df.to_csv(os.path.join(save_dir, "standardization_parameters.csv"), index=False)


print("\n=== Event counts ===")
print(df[event_col].value_counts(dropna=False))
print("\n=== Sex × event ===")
print(pd.crosstab(df[sex_col], df[event_col], margins=True, dropna=False))
print("\n=== Stage × event ===")
print(pd.crosstab(df["stage_group"], df[event_col], margins=True, dropna=False))

quality_df = pd.DataFrame({
    "variable": [
        duration_col,
        event_col,
        age_col,
        sex_col,
        stage_col,
        risk_col,
        edge_col,
    ],
    "missing_n": [
        df[duration_col].isna().sum(),
        df[event_col].isna().sum(),
        df[age_col].isna().sum(),
        df["sex_male"].isna().sum(),
        df["stage_advanced"].isna().sum(),
        df[risk_col].isna().sum(),
        df[edge_col].isna().sum(),
    ]
})

quality_df.to_csv(os.path.join(save_dir,"variable_quality_check.csv"), index=False)

print("\n=== Missing values ===")
print(quality_df)


display_names = {"age_per10": ("Age\n(per 10-year increase)"),
    "sex_male": ("Male vs female"),
    "stage_advanced": ("Advanced vs early stage"),
    "risk_z": ("Model risk score\n(per 1-SD increase)"),
    "tumor_necrosis_edge_z": ("Tumor–necrosis edge proportion\n""(per 1-SD increase)"),
}

In [ ]:
def plot_forest(results_df, save_path, title, model_order=None, figsize=(10, 6), width_ratios=(3.5, 4.0, 2.0, 1.0), factor_colors=None, row_spacing=1.20, group_gap=0.35):

    plot_df = results_df.copy()

    if model_order is not None:
        plot_df["model"] = pd.Categorical(plot_df["model"], categories=model_order, ordered=True)
        plot_df = (plot_df.sort_values("model").reset_index(drop=True))

    plot_rows = []

    for model_name, group in plot_df.groupby("model", sort=False, observed=True):
        plot_rows.append({"is_header": True, "display_name": model_name})
        for _, row in group.iterrows():
            d = row.to_dict()
            d["is_header"] = False
            plot_rows.append(d)
    rows_df = pd.DataFrame(plot_rows)
    n_rows = len(rows_df)


    raw_y = []
    current_y = 0.0

    for _, row in rows_df.iterrows():

        raw_y.append(current_y)

        if row["is_header"]:
            current_y += row_spacing
        else:
            current_y += row_spacing

            idx = len(raw_y) - 1

            if (idx + 1 < len(rows_df) and rows_df.iloc[idx + 1]["is_header"]):
                current_y += group_gap

    max_y = max(raw_y)
    y_positions = np.array([max_y - y for y in raw_y])
    total_y_height = max_y + row_spacing
    fig_height = max(figsize[1], total_y_height * 0.50)

    valid = plot_df[(plot_df["ci95_low"] > 0) & (plot_df["ci95_high"] > 0)]
    x_min = max(valid["ci95_low"].min() * 0.75, 0.1)
    x_max = (valid["ci95_high"].max()* 1.25)


    if factor_colors is None:

        factor_colors = {
            "Age\n(per 10-year increase)": "#5B8DB8",
            "Male vs female": "#C58A52",
            "Advanced vs early stage": "#67A567",
            "Model risk score\n(per 1-SD increase)": "#B66565",
            "Tumor-necrosis edge proportion\n(per 1-SD increase)": "#8873A8",
        }

    default_color = "#7A7A7A"


    fig = plt.figure(figsize=(figsize[0], fig_height))
    gs = fig.add_gridspec(
        nrows=1,
        ncols=4,
        width_ratios=width_ratios,
        wspace=0.02
    )

    ax_factor = fig.add_subplot(gs[0, 0])
    ax_forest = fig.add_subplot(gs[0, 1], sharey=ax_factor)
    ax_hr = fig.add_subplot(gs[0, 2], sharey=ax_factor)
    ax_p = fig.add_subplot(gs[0, 3], sharey=ax_factor)
    axes = [ax_factor, ax_forest, ax_hr, ax_p]

    y_bottom = -row_spacing * 0.7
    y_top = y_positions[0] + row_spacing * 1.2

    for ax in axes:
        ax.set_ylim(y_bottom, y_top)
        ax.set_yticks([])

    for row_idx, (_, row) in enumerate(rows_df.iterrows()):
        y = y_positions[row_idx]
        if row["is_header"]:
            ax_factor.text(
                0.00,
                y,
                row["display_name"],
                transform=ax_factor.get_yaxis_transform(),
                ha="left",
                va="center",
                fontsize=10,
                fontweight="bold"
            )

        else:
            ax_factor.text(
                0.00,
                y,
                row["display_name"],
                transform=ax_factor.get_yaxis_transform(),
                ha="left",
                va="center",
                fontsize=10
            )


    ax_forest.axvline(1, linestyle="--", linewidth=1, color="gray", zorder=1)

    for row_idx, (_, row) in enumerate(rows_df.iterrows()):

        if row["is_header"]:
            continue

        y = y_positions[row_idx]

        hr = float(row["hazard_ratio"])
        low = float(row["ci95_low"])
        high = float(row["ci95_high"])

        factor_name = row["display_name"]

        color = factor_colors.get(factor_name,default_color)

        ax_forest.plot(
            [low, high],
            [y, y],
            color=color,
            linewidth=1.2,
            alpha=0.75,
            zorder=2
        )

        ax_forest.scatter(
            hr,
            y,
            s=45,
            marker="D",
            color=color,
            edgecolor="none",
            zorder=3
        )


    ax_forest.set_xlim(x_min, x_max)
    ax_forest.set_xticks([0.5, 1, 2])
    ax_forest.set_xticklabels(["0.5", "1", "2"])
    ax_forest.set_xlabel("Hazard ratio", fontsize=10)
    ax_forest.grid(axis="x", linestyle=":", linewidth=0.7, alpha=0.35)

    ax_hr.text(
        0.02,
        0.97,
        "HR (95% CI)",
        transform=ax_hr.transAxes,
        ha="left",
        va="top",
        fontsize=10,
        fontweight="bold"
    )

    for row_idx, (_, row) in enumerate(rows_df.iterrows()):

        if row["is_header"]:
            continue

        y = y_positions[row_idx]

        hr_text = (
            f"{row['hazard_ratio']:.3f} "
            f"({row['ci95_low']:.3f}–"
            f"{row['ci95_high']:.3f})"
        )

        ax_hr.text(
            0.02,
            y,
            hr_text,
            transform=ax_hr.get_yaxis_transform(),
            ha="left",
            va="center",
            fontsize=9.5
        )


    ax_p.text(
        0.05,
        0.97,
        "P",
        transform=ax_p.transAxes,
        ha="left",
        va="top",
        fontsize=10,
        fontweight="bold")

    for row_idx, (_, row) in enumerate(rows_df.iterrows()):

        if row["is_header"]:
            continue

        y = y_positions[row_idx]

        ax_p.text(
            0.05,
            y,
            format_p(row["p_value"]),
            transform=ax_p.get_yaxis_transform(),
            ha="left",
            va="center",
            fontsize=9.5
        )


    for row_idx, (_, row) in enumerate(rows_df.iterrows()):

        if not row["is_header"]:
            continue

        y = y_positions[row_idx]

        separator_y = y - row_spacing * 0.55

        for ax in axes:
            ax.axhline(
                separator_y,
                linewidth=0.6,
                alpha=0.2,
                color="gray"
            )


    for ax in [ax_factor, ax_hr, ax_p]:
        ax.set_xlim(0, 1)
        ax.set_xticks([])
        for spine in ax.spines.values():
            spine.set_visible(False)

    for spine in ["top", "right", "left"]:
        ax_forest.spines[spine].set_visible(False)

    fig.suptitle(title, fontsize=10, y=0.98)


    plt.subplots_adjust(
        left=0.04,
        right=0.98,
        top=0.90,
        bottom=0.10,
        wspace=0.02
    )

    plt.savefig(save_path, dpi=600, bbox_inches="tight")

    print(f"[Saved] {save_path}")

    plt.show()

In [ ]:
univariable_variables = [
    "age_per10",
    "sex_male",
    "stage_advanced",
    "risk_z",
    "tumor_necrosis_edge_z",
]

univariable_results = []

for variable in univariable_variables:
    cph, result_df, model_df = fit_cox(
        df=df,
        covariates=[variable],
        model_name="Univariable Cox",
        display_names=display_names,
        penalizer=0.0,
    )

    univariable_results.append(result_df)

univariable_df = pd.concat(univariable_results, ignore_index=True)
univariable_csv = os.path.join(save_dir, "univariable_selected_predictors.csv")
univariable_df.to_csv(univariable_csv, index=False)
plot_forest(
    results_df=univariable_df,
    save_path=os.path.join(save_dir, "univariable_selected_predictors_forest.png"),
    title="Univariable Cox regression of clinical and model-derived prognostic factors",
    model_order=["Univariable Cox"],
    figsize=(10, 4)
)

In [ ]:
model0_covariates = ["age_per10", "sex_male", "stage_advanced"]
cph0, result0, model0_df = fit_cox(
    df=df,
    covariates=model0_covariates,
    model_name=("Clinic"),
    display_names=display_names,
    penalizer=0.0,
)

result0.to_csv(os.path.join(save_dir, "multivariable_clinical.csv"), index=False)
check_ph(cph0, model0_df, "Clinic")

model1_covariates = [
    "age_per10",
    "sex_male",
    "stage_advanced",
    "tumor_necrosis_edge_z",
]

cph1, result1, model1_df = fit_cox(
    df=df,
    covariates=model1_covariates,
    model_name=("Clinic + Edge proportion"),
    display_names=display_names,
    penalizer=0.0,
)

result1.to_csv(os.path.join(save_dir, "multivariable_clinical_plus_edge.csv"), index=False)
check_ph(cph1, model1_df, "Clinic + Edge ratio")


model2_covariates = [
    "age_per10",
    "sex_male",
    "stage_advanced",
    "risk_z",
]

cph2, result2, model2_df = fit_cox(
    df=df,
    covariates=model2_covariates,
    model_name=("Clinic + Risk score"),
    display_names=display_names,
    penalizer=0.0,
)

result2.to_csv(os.path.join(save_dir, "multivariable_clinical_plus_risk.csv"), index=False)

check_ph(cph2, model2_df, "Clinic + Risk score")


model3_covariates = [
    "age_per10",
    "sex_male",
    "stage_advanced",
    "risk_z",
    "tumor_necrosis_edge_z",
]

cph3, result3, model3_df = fit_cox(
    df=df,
    covariates=model3_covariates,
    model_name=("Clinic + Risk score + Edge proportion"),
    display_names=display_names,
    penalizer=0.0,
)

result3.to_csv(os.path.join(save_dir,"multivariable_clinical_plus_risk_plus_edge.csv"),index=False)

check_ph(cph3, model3_df, "Clinic + Risk + Edge")


multivariable_df = pd.concat([result0, result1, result2, result3], ignore_index=True)
multivariable_df.to_csv(os.path.join(save_dir, "all_multivariable_results.csv"), index=False)
plot_forest(
    results_df=multivariable_df,
    save_path=os.path.join(save_dir,"multivariable_forest.png"),
    title="Multivariable Cox regression of clinical and model-derived prognostic factors",
    model_order=[
        ("Clinic"),
        ("Clinic + Edge proportion"),
        ("Clinic + Risk score"),
        ("Clinic + Risk score + Edge proportion"),
    ],
    figsize=(10, 8)
)

In [ ]:
model_summary = pd.DataFrame([
    {
        "model": ("Clinical factors"),
        "n": len(model0_df),
        "events": int(model0_df[event_col].sum()),
        "c_index": cph0.concordance_index_,
        "partial_AIC": cph0.AIC_partial_,
        "log_likelihood": cph0.log_likelihood_,
    },
    {
        "model": ("Clinical factors + tumor–necrosis edge proportion"),
        "n": len(model1_df),
        "events": int(model1_df[event_col].sum()),
        "c_index": cph1.concordance_index_,
        "partial_AIC": cph1.AIC_partial_,
        "log_likelihood": cph1.log_likelihood_,
    },
    {
        "model": ("Clinical factors + model risk score"),
        "n": len(model2_df),
        "events": int(model2_df[event_col].sum()),
        "c_index": cph2.concordance_index_,
        "partial_AIC": cph2.AIC_partial_,
        "log_likelihood": cph2.log_likelihood_,
    },
    {
        "model": ("Clinical factors + risk score + tumor–necrosis edge proportion"),
        "n": len(model3_df),
        "events": int(model3_df[event_col].sum()),
        "c_index": cph3.concordance_index_,
        "partial_AIC": cph3.AIC_partial_,
        "log_likelihood": cph3.log_likelihood_,
    },
])

model_summary.to_csv(os.path.join(save_dir,"multivariable_model_summary.csv"),index=False)

print("\n=== Model summary ===")
print(model_summary.round(4).to_string(index=False))

print("\nFinished.")
print("Results saved in:")
print(save_dir)

In [ ]:
cox_csv_path = "/root/Desktop/data/private/hjx_product/results_pgexplainer_0614/CombinedModel_ss/statistic_analysis/cox_analysis/_results.csv"
save_dir = "/root/Desktop/data/private/hjx_product/results_pgexplainer_0614/CombinedModel_ss/statistic_analysis/cox_analysis/"
os.makedirs(save_dir, exist_ok=True)
save_png = os.path.join(save_dir, "univariable_cox_forest_plot.png")

df = pd.read_csv(cox_csv_path)
print("Available columns:")
print(df.columns.tolist())

def find_column(dataframe, candidates):
    for col in candidates:
        if col in dataframe.columns:
            return col

    raise ValueError(
        f"Cannot find any of these columns: {candidates}\n"
        f"Available columns: {dataframe.columns.tolist()}"
    )

feature_col = find_column(df,["feature", "Feature", "variable", "Variable"])
hr_col = find_column(df,["hazard_ratio", "HR", "hr", "exp(coef)"])
ci_low_col = find_column(df,["ci95_low","CI_low","ci_low","lower_95","exp(coef) lower 95%"])
ci_high_col = find_column(df,["ci95_high","CI_high","ci_high","upper_95","exp(coef) upper 95%"])
p_col = find_column(df, ["p_value", "p", "P", "pvalue"])
fdr_col = find_column(df, ["p_adj_BH", "FDR", "fdr", "adjusted_p"])


for col in [hr_col, ci_low_col, ci_high_col, p_col, fdr_col]:
    df[col] = pd.to_numeric(df[col],errors="coerce")

df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=[feature_col,hr_col,ci_low_col,ci_high_col,p_col,fdr_col]).copy()

df = df[(df[hr_col] > 0) & (df[ci_low_col] > 0) & (df[ci_high_col] > 0)].copy()


def classify_feature(feature_name):
    name = str(feature_name)

    if "_edge_ratio" in name or "_edge_weight_ratio" in name:
        return "Tissue interactions"

    if name.endswith("_delta"):
        return "Relative tissue composition"

    if name.startswith("subgraph_"):
        return "Pixel-level tissue composition"

    if name.endswith("_ratio"):
        return "Patch-level tissue composition"

    return "Other"

df["category"] = df[feature_col].apply(classify_feature)


def prettify_feature_name(feature_name):
    name = str(feature_name)

    replacements = {
        "subgraph_": "",
        "_pixel_ratio": "",
        "_edge_weight_ratio": " edge weight",
        "_edge_ratio": " edge ratio",
        "_delta": "",
        "_ratio": "",
        "necrosis": "Necrosis",
        "normal": "Normal",
        "inflammation": "Inflammation",
        "tumor": "Tumor",
        "steatosis": "Steatosis",
        "Fibrous": "Fibrosis",
        "Reactive": "Bile duct reaction",
        "__": "–",
        "_": " ",
    }

    for old, new in replacements.items():
        name = name.replace(old, new)

    words = name.split()

    formatted_words = []

    for word in words:
        if word.upper() == "WSI":
            formatted_words.append("WSI")
        else:
            formatted_words.append(word)

    return " ".join(formatted_words)


df["display_name"] = df[feature_col].apply(prettify_feature_name)


category_order = [
    "Patch-level tissue composition",
    "Pixel-level tissue composition",
    "Relative tissue composition",
    "Tissue interactions",
    "Other",
]

df["category"] = pd.Categorical(df["category"],categories=category_order,ordered=True)

df = df.sort_values(["category", fdr_col, p_col]).reset_index(drop=True)

plot_rows = []

for category in category_order:
    category_df = df[df["category"] == category]

    if category_df.empty:
        continue

    plot_rows.append({
        "is_header": True,
        "display_name": category,
    })

    for _, row in category_df.iterrows():
        row_dict = row.to_dict()
        row_dict["is_header"] = False
        plot_rows.append(row_dict)

plot_df = pd.DataFrame(plot_rows)


def format_p(p):
    if pd.isna(p):
        return "NA"
    if p < 0.001:
        return "<0.001"

    return f"{p:.3f}"


def p_to_star(p):
    if pd.isna(p):
        return ""
    if p < 0.0001:
        return "****"
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"

    return ""


n_rows = len(plot_df)
width_ratios = (3.5, 4.5, 2.0, 1.0, 1.0)
fig_height = max(7, n_rows * 0.43)
fig = plt.figure(figsize=(10, fig_height))
gs = fig.add_gridspec(
    nrows=1,
    ncols=5,
    width_ratios=width_ratios,
    wspace=0.03
)

ax_factor = fig.add_subplot(gs[0, 0])
ax_forest = fig.add_subplot(gs[0, 1], sharey=ax_factor)
ax_hr = fig.add_subplot(gs[0, 2], sharey=ax_factor)
ax_p = fig.add_subplot(gs[0, 3], sharey=ax_factor)
ax_fdr = fig.add_subplot(gs[0, 4], sharey=ax_factor)
axes = [ax_factor, ax_forest, ax_hr, ax_p, ax_fdr]

y_positions = np.arange(n_rows)[::-1]


valid_df = df[(df[ci_low_col] > 0) & (df[ci_high_col] > 0)].copy()

valid_low = valid_df[ci_low_col].min()
valid_high = valid_df[ci_high_col].max()

x_min = max(valid_low * 0.75, 0.05)
x_max = valid_high * 1.25

for ax in axes:
    ax.set_ylim(-1, n_rows)
    ax.set_yticks([])

for row_idx, (_, row) in enumerate(plot_df.iterrows()):
    y = y_positions[row_idx]
    if row["is_header"]:
        ax_factor.text(
            0.00,
            y,
            row["display_name"],
            transform=ax_factor.get_yaxis_transform(),
            fontsize=10,
            fontweight="bold",
            va="center",
            ha="left"
        )
        continue

    fdr = float(row[fdr_col])
    is_fdr_sig = fdr < 0.05
    ax_factor.text(
        0.00,
        y,
        row["display_name"],
        transform=ax_factor.get_yaxis_transform(),
        fontsize=10,
        fontweight=("bold" if is_fdr_sig else "normal"),
        va="center",
        ha="left"
    )


ax_forest.axvline(
    1.0,
    linestyle="--",
    linewidth=1.0,
    color="gray",
    zorder=1
)

for row_idx, (_, row) in enumerate(plot_df.iterrows()):

    y = y_positions[row_idx]

    if row["is_header"]:
        continue

    hr = float(row[hr_col])
    ci_low = float(row[ci_low_col])
    ci_high = float(row[ci_high_col])
    p_value = float(row[p_col])
    fdr = float(row[fdr_col])

    is_fdr_sig = fdr < 0.05

    marker_size = 6
    marker = "D"

    line_width = 1.2

    alpha = 0.75

    # CI
    ax_forest.plot(
        [ci_low, ci_high],
        [y, y],
        linewidth=line_width,
        alpha=alpha,
        zorder=2
    )

    # HR point
    ax_forest.scatter(
        hr,
        y,
        s=marker_size ** 2,
        marker=marker,
        zorder=2,
        alpha=alpha
    )

    star = p_to_star(p_value)

    if star:
        ax_forest.text(
            ci_high * 1.03,
            y,
            star,
            fontsize=9,
            va="center",
            ha="left"
        )


header_y = y_positions[0] + 0.4

ax_hr.text(
    0.02,
    header_y,
    "HR (95% CI)",
    transform=ax_hr.get_yaxis_transform(),
    fontsize=10,
    fontweight="bold",
    ha="left",
    va="bottom"
)

ax_p.text(
    0.05,
    header_y,
    "P",
    transform=ax_p.get_yaxis_transform(),
    fontsize=10,
    fontweight="bold",
    ha="left",
    va="bottom"
)

ax_fdr.text(
    0.05,
    header_y,
    "FDR",
    transform=ax_fdr.get_yaxis_transform(),
    fontsize=10,
    fontweight="bold",
    ha="left",
    va="bottom"
)


for row_idx, (_, row) in enumerate(plot_df.iterrows()):

    if row["is_header"]:
        continue

    y = y_positions[row_idx]

    hr = float(row[hr_col])
    ci_low = float(row[ci_low_col])
    ci_high = float(row[ci_high_col])
    p_value = float(row[p_col])
    fdr = float(row[fdr_col])

    is_fdr_sig = fdr < 0.05

    font_weight = (
        "bold"
        if is_fdr_sig
        else "normal"
    )

    # HR (95% CI)
    ax_hr.text(
        0.02,
        y,
        f"{hr:.3f} ({ci_low:.3f}–{ci_high:.3f})",
        transform=ax_hr.get_yaxis_transform(),
        fontsize=9.5,
        fontweight=font_weight,
        ha="left",
        va="center"
    )

    # P
    ax_p.text(
        0.05,
        y,
        format_p(p_value),
        transform=ax_p.get_yaxis_transform(),
        fontsize=9.5,
        fontweight=font_weight,
        ha="left",
        va="center"
    )

    # FDR
    ax_fdr.text(
        0.05,
        y,
        format_p(fdr),
        transform=ax_fdr.get_yaxis_transform(),
        fontsize=9.5,
        fontweight=font_weight,
        ha="left",
        va="center"
    )


for row_idx, (_, row) in enumerate(plot_df.iterrows()):

    if not row["is_header"]:
        continue

    y = y_positions[row_idx]

    for ax in axes:

        ax.axhline(
            y - 0.48,
            linewidth=0.5,
            alpha=0.25,
            color="gray"
        )


ax_forest.set_xlim(x_min, x_max)
ax_forest.set_xticks([0.5, 1, 2])

ax_forest.set_xticklabels(["0.5", "1", "2"], fontsize=9)
ax_forest.set_xlabel("Hazard ratio", fontsize=10)
ax_forest.grid(
    axis="x",
    linestyle=":",
    linewidth=0.7,
    alpha=0.35
)


for ax in [ax_factor, ax_hr, ax_p, ax_fdr]:
    ax.set_xlim(0, 1)
    ax.set_xticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)

for spine in ["top", "right", "left"]:
    ax_forest.spines[spine].set_visible(False)


fig.suptitle(
    "Univariable Cox analysis of graph-derived histopathological features",
    fontsize=10,
    y=0.98
)


plt.subplots_adjust(
    left=0.04,
    right=0.985,
    top=0.92,
    bottom=0.10,
    wspace=0.03
)


plt.savefig(save_png, dpi=600, bbox_inches="tight")

print(f"[Saved PNG] {save_png}")

plt.show()